# 02 — Silver: features das transações

Espelha `RETREINO/00_struct_table.py`: instala `haversine`, registra `dist_haversine` e deriva as features em Spark SQL. Adapta a origem para `desloc.bronze.transacoes`, grava a tabela permanente `desloc.silver.transacoes` e chama a distância calculada de `distancia_lalo`.

A regra de período segue sua definição anterior: 1 = 07h–11h; 2 = 12h–17h; 3 = demais horários. A data/hora completa não é gravada na Silver.

In [ ]:
!pip install haversine

In [ ]:
%restart_python

In [ ]:
from haversine import haversine
from pyspark.sql.types import FloatType

def dist_haversine(lat_o, lon_o, lat_d, lon_d):
    return haversine((lat_o, lon_o), (lat_d, lon_d))

spark.udf.register("dist_haversine", dist_haversine, FloatType())

In [ ]:
spark.sql("""
    CREATE OR REPLACE TABLE desloc.silver.transacoes AS
    WITH tbl_features AS (
    SELECT
        day(to_timestamp(datahora_entregue, 'dd/MM/yyyy HH:mm')) as dia,
        dayofweek(to_timestamp(datahora_entregue, 'dd/MM/yyyy HH:mm')) as dia_semana,
        month(to_timestamp(datahora_entregue, 'dd/MM/yyyy HH:mm')) as mes,
        hour(to_timestamp(datahora_entregue, 'dd/MM/yyyy HH:mm')) as hora,
        duracao,
        distancia,
        vlr_pago,
        latitude_origem,
        longitude_origem,
        latitude_destino,
        longitude_destino
    FROM desloc.bronze.transacoes
    )
    SELECT
    dia,
    dia_semana,
    mes,
    hora,
    case when hora < 12 and hora > 6 then 1
        when hora >= 12 and hora < 18 then 2
        else 3 end as periodo,
    duracao,
    vlr_pago,
    latitude_origem,
    longitude_origem,
    latitude_destino,
    longitude_destino,
    dist_haversine(latitude_origem, longitude_origem, latitude_destino, longitude_destino) as distancia_lalo
    FROM tbl_features
""")

## Conferência
A consulta abaixo apenas mostra uma amostra da tabela permanente criada acima.

In [ ]:
display(spark.sql("SELECT * FROM desloc.silver.transacoes LIMIT 5"))

## Limite para previsão antecipada
`dia`, `dia_semana`, `mes` e `hora` derivam do horário de entrega. Para cotar o preço antes da corrida, é necessária a data/hora disponível na solicitação ou coleta.